***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.8 离散傅里叶变换与 FFT：从解析公式到可计算算法](2_8_the_discrete_fourier_transform.ipynb)
    * 下一节：[2.10 线性代数：从测量方程到矩阵表示](2_10_linear_algebra.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.9 采样理论：离散测量的分辨率与混叠<a id='math:sec:sampling_theory'></a>

采样理论回答一个根本问题：从有限个离散测量值中，究竟能恢复多少关于连续信号的信息。对射电干涉测量而言，这个问题就是日常现实：阵列只在有限时间、有限频率通道和有限 `uv` 点上取得带噪样本，却希望重建连续天空亮度分布。

第 2.8 节说明 DFT 默认有限周期模型；本节进一步说明这个模型背后的信息限制。采样间隔决定可无歧义表示的最高频率，总观测范围决定频率分辨率，有限窗口产生谱泄漏，高于 Nyquist 频率的内容会折返回低频形成混叠。这些现象共同决定了哪些结构可信、哪些结构可能只是采样伪影。


### 2.9.1 采样算子：comb 与有限窗口<a id='math:sec:sampling_continuous_function'></a>

理想等间隔采样可以写成连续函数与 Dirac comb 的乘积。若采样间隔为 $\Delta t$，则

<a id='math:eq:ideal_sampling_operator'></a><!--\label{math:eq:ideal_sampling_operator}-->
$$
y_{\rm samp}(t)=y(t)\sum_{n=-\infty}^{+\infty}\delta(t-n\Delta t)
=\sum_{n=-\infty}^{+\infty}y(n\Delta t)\delta(t-n\Delta t).
$$

真实观测还必须乘以有限窗口 $w(t)$，例如只在长度为 $T$ 的区间内取样：

$$
y_{\rm obs}(t)=w(t)y(t)\sum_n\delta(t-n\Delta t).
$$

这个写法把两类效应分开了。comb 由 $\Delta t$ 决定，它控制频谱副本的间隔和 Nyquist 频率；窗口由总时长 $T$ 决定，它控制频域分辨率和谱泄漏。二者经常同时出现，但物理含义不同。


有限窗口的影响可以先从一个简单正弦看出来。即使信号只有一个真实频率，有限时长观测也等价于把无限正弦乘以一个矩形窗。由第 2.7 节乘积定理，频域中会出现与 `sinc` 型窗口函数的卷积，因此谱峰具有有限宽度和旁瓣。总时长越短，`sinc` 主瓣越宽，频率分辨率越差。

![有限窗口与频率分辨率](figures/sampling_window_leakage_resolution.png)

**图 2.9.1** 有限观测窗口导致谱峰展宽。采样率足够高时，增加总观测时长 $T$ 会让频率间隔 $\Delta f=1/T$ 变小，谱峰更窄；这与 Nyquist 频率由采样间隔决定是两件不同的事。


### 2.9.2 Nyquist 条件：频谱副本不能重叠<a id='math:sec:nyquists_sampling_theorem'></a>

采样间隔为 $\Delta t$ 时，采样频率为

$$
f_s=\frac{1}{\Delta t},
$$

Nyquist 频率为

<a id='math:eq:nyquist_frequency'></a><!--\label{math:eq:nyquist_frequency}-->
$$
f_N=\frac{f_s}{2}=\frac{1}{2\Delta t}.
$$

采样定理常被表述为“采样频率至少要大于最高信号频率的两倍”。更本质的说法是：采样会使连续频谱以 $f_s$ 为间隔周期复制；只要这些副本不重叠，原始带限信号就可以由样本唯一恢复；一旦副本重叠，不同真实频率在采样数据中变得不可区分。

泊松求和公式给出这个图像的数学表达：

<a id='math:eq:sampling_spectrum_replication'></a><!--\label{math:eq:sampling_spectrum_replication}-->
$$
\sum_{n=-\infty}^{+\infty}\Delta t\,y(n\Delta t)e^{-2\pi ifn\Delta t}
=\sum_{k=-\infty}^{+\infty}Y(f-kf_s).
$$

左边由离散样本构成，右边是连续频谱的周期副本。

![频谱副本与 Nyquist 条件](figures/sampling_spectrum_replicas_aliasing.png)

**图 2.9.2** 采样造成的频谱副本。带限信号的副本彼此分离时，可以恢复中央主带；副本重叠时，发生混叠。


这里必须区分两个尺度。增大采样间隔 $\Delta t$ 会降低 Nyquist 频率，使高频内容更容易混叠；缩短总观测时长 $T=N\Delta t$ 则会增大频率间隔 $\Delta f=1/T$，使频谱分辨率变差。前者决定“最高能无歧义表示到哪里”，后者决定“两个相近频率能否分开”。


### 2.9.3 混叠：高频伪装成低频<a id='math:sec:aliasing'></a>

混叠不是简单的高频丢失，而是高于 Nyquist 频率的成分折返回主带，与真实低频成分混在一起。对采样率 $f_s$ 的实信号，一个频率 $f$ 在主带中的折返频率可写为

<a id='math:eq:alias_frequency'></a><!--\label{math:eq:alias_frequency}-->
$$
f_{\rm alias}=\left|\left[(f+f_s/2)\bmod f_s\right]-f_s/2\right|.
$$

例如 $f_s=10\,\mathrm{Hz}$ 时，$7\,\mathrm{Hz}$ 的余弦在采样点上与 $3\,\mathrm{Hz}$ 的余弦完全相同。离散数据本身无法判断样本来自哪一个连续频率。

![混叠折返](figures/sampling_alias_frequency_fold.png)

**图 2.9.3** 混叠折返。不同连续频率可能在相同采样时刻产生完全一样的样本，因此采样之后再尝试区分它们已经太晚。


抗混叠滤波必须发生在采样之前。它的作用不是保留所有高频信息，而是在采样前主动去掉 Nyquist 频率以上的内容，避免这些内容折返污染主带。采样之后看到的伪低频成分，形式上与真实低频成分没有区别，不能靠简单后处理完全恢复。


### 2.9.4 sinc 重建与采样定理的理想前提<a id='math:sec:sinc_reconstruction'></a>

若信号严格带限在 $|f|<f_N$，并且拥有无限长、无噪声、等间隔样本，则连续信号可由 Whittaker-Shannon 插值公式重建：

<a id='math:eq:whittaker_shannon'></a><!--\label{math:eq:whittaker_shannon}-->
$$
y(t)=\sum_{n=-\infty}^{+\infty}y(n\Delta t)
\operatorname{sinc}\left(\frac{t-n\Delta t}{\Delta t}\right).
$$

这个公式来自频域中对中央主带的理想矩形截取，而矩形窗的反变换正是 `sinc`。它给出的是理论极限，不是现实数据处理中可以无条件照搬的算法。真实观测有限长、带噪，且信号通常不可能严格带限，因此实际重建总要面对截断、窗口、噪声和模型假设。

![sinc 重建](figures/sampling_sinc_reconstruction.png)

**图 2.9.4** `sinc` 重建的前提。满足 Nyquist 条件时，理想插值可以恢复连续曲线；欠采样时，插值公式只能重建被混叠污染后的信号。


### 2.9.5 干涉测量中的采样直觉<a id='math:sec:sampling_interferometry'></a>

把时间频率换成天空角坐标与空间频率，采样理论的逻辑保持不变。`uv` 平面采样间隔与图像视场相关，最长基线决定可测的最高空间频率并限制角分辨率，最短基线决定对大尺度结构的敏感性。有限 `uv` 覆盖相当于在空间频率平面乘以采样函数，于是图像域中出现脏波束卷积和旁瓣结构。

这也解释了为什么成像伪影常常不是“软件画错了”，而是有限采样的数学后果。不完整 `uv` 覆盖会让某些空间尺度缺失；过粗网格或不恰当视场会引入周期边界和混叠；权重和 taper 会改变分辨率与旁瓣之间的折中。后续成像章节中的许多技术选择，本质上都是在管理这些采样限制。


### 2.9.6 本节要点

采样间隔 $\Delta t$ 决定 Nyquist 频率，总观测时长 $T$ 决定频率分辨率，有限窗口导致谱泄漏，欠采样导致混叠。采样定理的核心不是一句“两倍最高频率”的口号，而是频谱副本是否重叠。对射电干涉测量而言，有限 `uv` 采样带来的分辨率、最大可恢复尺度、旁瓣和伪影，都是这一数学结构在空间频率平面上的表现。


***

* 下一节：[2.10 线性代数：从测量方程到矩阵表示](2_10_linear_algebra.ipynb)
